# Question 2: Computational Thinking in Academic Systems

## Introduction

Computational thinking is a problem-solving methodology built on four pillars: **decomposition**, **pattern recognition**, **abstraction**, and **algorithm design**. Academic systems — which manage student registration, marks, and grading — are a natural fit for this approach because they involve large, repetitive, rule-based workflows.

## (a) Applying Computational Thinking

**Decomposition** — the academic system is broken into independent sub-problems:
1. Student Registration
2. Marks Entry
3. Grade Calculation
4. Reporting

**Pattern Recognition** — every student follows the identical workflow:

```
Registration → Marks Entry → Average Calculation → Grade Assignment → Reporting
```

Because this pattern repeats for every student, **one algorithm** (written once) can be reused for the entire student body instead of writing custom logic per student.

**Abstraction** — the system exposes only what matters to its users (Student ID, Name, Marks) and hides irrelevant implementation detail (how records are stored on disk, how the mean is computed internally, UI rendering, etc.).

## (b) Algorithm to Calculate Average Marks and Grade

The algorithm takes a list of marks, computes the arithmetic mean, and maps it onto mutually-exclusive grade bands. Input validation is added (empty list, out-of-range marks) — a gap in the original draft, since dividing by zero on an empty list, or accepting a mark of `150`, would silently corrupt results.

In [1]:
def calculate_grade(marks):
    """Compute (average, grade) for a list of numeric marks in [0, 100]."""
    if not marks:
        raise ValueError("marks list cannot be empty")
    for m in marks:
        if not (0 <= m <= 100):
            raise ValueError(f"invalid mark: {m} (must be 0-100)")

    average = sum(marks) / len(marks)

    if average >= 80:
        grade = "A"
    elif average >= 70:
        grade = "B"
    elif average >= 60:
        grade = "C"
    elif average >= 50:
        grade = "D"
    else:
        grade = "F"

    return average, grade


marks = [75, 80, 90]
avg, grade = calculate_grade(marks)
print(f"Average: {avg:.2f}")
print(f"Grade: {grade}")

Average: 81.67
Grade: A


## (c) Pseudocode

```text
START

INPUT marks[]

IF marks is empty THEN
    DISPLAY "Error: No marks provided"
    STOP
END IF

FOR EACH mark IN marks
    IF mark < 0 OR mark > 100 THEN
        DISPLAY "Error: Invalid mark"
        STOP
    END IF
END FOR

average ← SUM(marks) / COUNT(marks)

IF average >= 80 THEN
    grade ← "A"
ELSE IF average >= 70 THEN
    grade ← "B"
ELSE IF average >= 60 THEN
    grade ← "C"
ELSE IF average >= 50 THEN
    grade ← "D"
ELSE
    grade ← "F"
END IF

DISPLAY average, grade

END
```

## (d) Algorithm Correctness and Validation

**Correctness argument:**
- Every mark in the input list contributes to the sum **exactly once** → the mean is computed correctly.
- The grade bands `[80,∞) [70,80) [60,70) [50,60) [0,50)` are **mutually exclusive and exhaustive** (they partition the real number line over the valid mark range) → exactly one grade is always assigned, for any valid average.
- The algorithm operates on a **finite list** and contains no loops with unbounded conditions → it always terminates.

This satisfies the two pillars of algorithm correctness: **partial correctness** (if it terminates, the output is right) and **termination** (it always terminates) — together giving **total correctness**.

**Validation methods**, demonstrated below as an executable test suite rather than just claimed in prose:
- **Unit testing** — exact known input/output pairs
- **Boundary testing** — values exactly at grade-band edges (79 vs 80, 49 vs 50, etc.)
- **Input validation / exception handling** — invalid input (empty list, out-of-range marks) is rejected, not silently miscomputed

In [2]:
# ---- Automated validation suite ----

# Unit tests
assert calculate_grade([80, 80, 80]) == (80.0, "A")
assert calculate_grade([75, 80, 90])[1] == "A"   # mean = 81.67

# Boundary tests (the classic source of off-by-one bugs in grading systems)
assert calculate_grade([80, 80, 80])[1] == "A"
assert calculate_grade([79, 79, 79])[1] == "B"
assert calculate_grade([70, 70, 70])[1] == "B"
assert calculate_grade([69, 69, 69])[1] == "C"
assert calculate_grade([60, 60, 60])[1] == "C"
assert calculate_grade([59, 59, 59])[1] == "D"
assert calculate_grade([50, 50, 50])[1] == "D"
assert calculate_grade([49, 49, 49])[1] == "F"

# Exception handling
try:
    calculate_grade([])
    raise AssertionError("should have raised on empty list")
except ValueError as e:
    print("Correctly rejected empty list:", e)

try:
    calculate_grade([150])
    raise AssertionError("should have raised on out-of-range mark")
except ValueError as e:
    print("Correctly rejected invalid mark:", e)

print("\nALL UNIT + BOUNDARY + EXCEPTION TESTS PASSED ✔")

Correctly rejected empty list: marks list cannot be empty
Correctly rejected invalid mark: invalid mark: 150 (must be 0-100)

ALL UNIT + BOUNDARY + EXCEPTION TESTS PASSED ✔


### Complexity Analysis

| Operation | Complexity |
|---|---|
| Summing marks | O(n) |
| Computing average | O(1) (given the sum) |
| Grade comparison | O(1) (fixed number of bands) |
| **Overall** | **O(n)**, where n = number of marks |

The algorithm scales linearly with the number of marks per student, which is acceptable since n is small and bounded (typically the number of courses a student takes per semester).